**Question** What exactly is one Aegis learning example, and how are its labels generated?
**Method** Reuse the repository generator, persist provenance, and export tensor contents to auditable CSV tables.
**Takeaway** The benchmark is a fixed-length pre-injection graph time series with continuous telemetry and explicit future-failure labels.

## Dataset contract

The generator observes 10 steps before the future threshold breach. Each active node has eight features: normalized CPU, memory, p95 latency, error rate, normalized in-degree, normalized out-degree, criticality, and normalized queue depth. The ten configured injectors are latency, CPU stress, service kill, memory leak, packet loss, thread exhaustion, database lock, slow query, message-queue lag, and cache down. Nominal episodes have no fault node.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from dataset.export_csv import export_csv
paths = export_csv()
paths

### What this tells us
The four files are generated from the tensor artifact and the persisted episode metadata. No result number is typed into the notebook; the returned paths confirm which files were actually written.

In [ ]:
import pandas as pd
from dataset.export_csv import FEATURE_NAMES
node = pd.read_csv('data/node_timeseries.csv')
episodes = pd.read_csv('data/episodes.csv')
flat = pd.read_csv('data/episodes_flat.csv')
print('features:', FEATURE_NAMES)
print('node_timeseries shape:', node.shape)
print('episodes shape:', episodes.shape)
print('flat shape:', flat.shape)
print(episodes[['fault_type','failure','split']].value_counts().sort_index())

### What this tells us
The shapes and class counts come directly from the exported data. The flat table contains per-node mean, maximum, slope, and neighbor-aggregate columns, while padded nodes are absent because the single-topology generator has no dummy slots.

In [ ]:
import matplotlib.pyplot as plt
example = node[node.episode_id == node.episode_id.iloc[0]]
fault = example[example.is_root_cause == 1].node_id.iloc[0] if example.is_root_cause.any() else example.node_id.iloc[0]
neighbor = example.node_id.iloc[1] if example.node_id.iloc[1] != fault else example.node_id.iloc[2]
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for feature, ax in zip(['cpu_utilization','p95_latency'], axes):
    for node_id, style in [(fault, '-'), (neighbor, '--')]:
        subset = example[example.node_id == node_id]
        ax.plot(subset.t, subset[feature], style, label=node_id)
    ax.set_ylabel(feature); ax.legend()
axes[-1].set_xlabel('observation step'); fig.tight_layout(); plt.show()

### What this tells us
This plot is a measured episode, not an illustrative curve. It lets us inspect the precursor trajectory of the selected root-cause node against another active node before any future-horizon label is revealed to a model.